# Agentic SABRE — .exe Feature Extractor

This notebook extracts the three feature vectors that Agentic SABRE uses for inference, **directly from a Windows PE (.exe) file**.

| Modality | Dimensions | How it is built |
|---|---|---|
| **Semantic** | 384-D | PE metadata → natural-language sentence → `all-MiniLM-L6-v2` embedding |
| **Behavioural** | 16-D | Static proxies → z-scores → rescaled into the trained scaler's space |
| **Knowledge** | 21-D | MITRE ATT&CK tactic + known-tool counts from strings and imports |

After extraction each vector is formatted exactly as the live orchestrator expects it,  
so you can pass the output straight to `orch.handle(x_sem, x_beh, x_know)`.

---
### Install dependencies
```bash
pip install pefile sentence-transformers numpy scipy
```

In [ ]:
import re, math, struct, string, hashlib, sys
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import pefile
from sentence_transformers import SentenceTransformer

print('All imports OK')

## 0 · Load the target executable

In [ ]:
EXE_PATH = Path("sample.exe")   # ← change to your file

assert EXE_PATH.exists(), f"File not found: {EXE_PATH}"
raw_bytes = EXE_PATH.read_bytes()
print(f"Loaded {EXE_PATH.name}  ({len(raw_bytes):,} bytes)")

pe = pefile.PE(str(EXE_PATH), fast_load=False)
print("PE parsed successfully")

---
## 1 · Semantic Feature Extraction (384-D)

The RDset pipeline converts each PE sample's properties into a natural-language sentence,  
then encodes it with `sentence-transformers/all-MiniLM-L6-v2` → 384-D float32 vector.

In [ ]:
# ── 1a. Low-level helpers ─────────────────────────────────────────────────────

def byte_entropy(data: bytes) -> float:
    if not data:
        return 0.0
    counts = Counter(data)
    total  = len(data)
    return -sum((c / total) * math.log2(c / total) for c in counts.values())

def extract_printable_strings(data: bytes, min_len: int = 6) -> list:
    printable = set(string.printable.encode())
    result, buf = [], []
    for b in data:
        if b in printable:
            buf.append(chr(b))
        else:
            if len(buf) >= min_len:
                result.append(''.join(buf))
            buf = []
    if len(buf) >= min_len:
        result.append(''.join(buf))
    return result

In [ ]:
# ── 1b. Extract all PE properties ─────────────────────────────────────────────

md5    = hashlib.md5(raw_bytes).hexdigest()
sha256 = hashlib.sha256(raw_bytes).hexdigest()

machine_map  = {0x014c: 'x86', 0x8664: 'x64', 0x01c0: 'ARM', 0xAA64: 'ARM64'}
subsystem_map = {2: 'GUI', 3: 'console', 1: 'native', 9: 'WinCE_GUI'}

machine      = machine_map.get(pe.FILE_HEADER.Machine, f'0x{pe.FILE_HEADER.Machine:04x}')
num_sections = pe.FILE_HEADER.NumberOfSections
subsystem    = subsystem_map.get(pe.OPTIONAL_HEADER.Subsystem, str(pe.OPTIONAL_HEADER.Subsystem))
image_base   = pe.OPTIONAL_HEADER.ImageBase
entry_point  = pe.OPTIONAL_HEADER.AddressOfEntryPoint
dll_chars    = pe.OPTIONAL_HEADER.DllCharacteristics
aslr         = bool(dll_chars & 0x0040)
nx           = bool(dll_chars & 0x0100)

sections = []
for s in pe.sections:
    name  = s.Name.rstrip(b'\x00').decode(errors='replace')
    chars = s.Characteristics
    sections.append({
        'name': name, 'entropy': round(s.get_entropy(), 3),
        'raw_size': s.SizeOfRawData, 'virt_size': s.Misc_VirtualSize,
        'virt_addr': s.VirtualAddress,
        'file_offset': s.PointerToRawData,
        'executable': bool(chars & 0x20000000),
        'writable':   bool(chars & 0x80000000),
    })

imports_by_dll = {}
if hasattr(pe, 'DIRECTORY_ENTRY_IMPORT'):
    for entry in pe.DIRECTORY_ENTRY_IMPORT:
        dll  = entry.dll.decode(errors='replace').lower()
        apis = [imp.name.decode(errors='replace') for imp in entry.imports if imp.name]
        imports_by_dll[dll] = apis

all_imports   = [api for apis in imports_by_dll.values() for api in apis]
imported_dlls = list(imports_by_dll.keys())
import_set    = set(all_imports)

exports = []
if hasattr(pe, 'DIRECTORY_ENTRY_EXPORT'):
    exports = [exp.name.decode(errors='replace')
               for exp in pe.DIRECTORY_ENTRY_EXPORT.symbols if exp.name]

all_strings = extract_printable_strings(raw_bytes, min_len=6)

print(f"Machine: {machine} | Subsystem: {subsystem} | Sections: {num_sections}")
print(f"Imported DLLs: {len(imported_dlls)}  |  Total imports: {len(all_imports)}")
print(f"Exports: {len(exports)}  |  Printable strings: {len(all_strings)}")

In [ ]:
# ── 1c. Build natural-language description and embed ──────────────────────────

SUSPICIOUS_APIS = [
    'CryptEncrypt','CryptDecrypt','CryptGenKey','CryptImportKey','CryptAcquireContext',
    'BCryptEncrypt','BCryptGenRandom','CreateFileW','CreateFileA','DeleteFileW',
    'MoveFileExW','FindFirstFileW','FindNextFileW','SetFileAttributesW',
    'CreateVssBackupComponents','CreateProcessW','CreateRemoteThread',
    'WriteProcessMemory','VirtualAllocEx','OpenProcess','TerminateProcess',
    'RegSetValueExW','RegCreateKeyExW','RegOpenKeyExW',
    'InternetOpenW','HttpSendRequestW','WSAStartup','connect',
    'WinHttpOpen','WinHttpSendRequest',
    'AdjustTokenPrivileges','LookupPrivilegeValueW','OpenProcessToken',
]

detected_sus = [api for api in SUSPICIOUS_APIS if api in import_set]

section_desc = '; '.join(
    f"section {s['name']} entropy {s['entropy']} size {s['raw_size']}"
    + (' executable' if s['executable'] else '')
    + (' writable'   if s['writable']   else '')
    for s in sections
)

pe_text = '; '.join(filter(None, [
    # File name and md5 are identifiers, not semantics, and are excluded as in
    # data_pipeline/rdset_semantic.py (IDENTIFIER_COLUMNS).
    f"architecture {machine}", f"subsystem {subsystem}",
    f"sections {num_sections}", section_desc,
    f"imports {len(all_imports)} from {len(imported_dlls)} dlls",
    f"dlls {' '.join(imported_dlls[:12])}",
    f"suspicious apis {' '.join(detected_sus[:20]) if detected_sus else 'none'}",
    f"exports {len(exports)}",
    f"aslr {'enabled' if aslr else 'disabled'}",
    f"nx {'enabled' if nx else 'disabled'}",
    f"entry point {hex(entry_point)}",
    f"image base {hex(image_base)}",
    f"strings sample {' '.join(s.replace(' ','_') for s in all_strings[:30])}",
]))

encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
encoder.max_seq_length = 256

sem_embedding = encoder.encode(pe_text, convert_to_numpy=True).astype(np.float32)
x_sem = {"embedding": sem_embedding}

print(f"Semantic embedding: {sem_embedding.shape}  range [{sem_embedding.min():.4f}, {sem_embedding.max():.4f}]")

---
## 2 · Behavioural Feature Extraction (16-D)

### Why this is hard
The 16 RanSMAP features are **runtime kernel telemetry** — bytes/s of actual disk I/O,  
variance of Logical Block Addresses actually accessed, entropy of memory pages actually  
written during execution. None of these exist in a static PE file.

### Strategy: z-score rescaling
From the trained `StandardScaler` we know the **mean and standard deviation** of each  
feature across the entire RanSMAP training set. The model never sees raw values — it sees  
z-scores. So instead of guessing the raw magnitude (which can be 10^10 for throughput),  
we:
1. Compute a **normalised signal** (0–1) from the PE for each feature  
2. Convert it to a **z-score** using domain knowledge of what high/low means  
3. Reconstruct the raw value: `raw = scaler.mean_[i] + z * scaler.scale_[i]`

This keeps the input in the distribution the model was trained on.  
The uncertainty the behavioural agent reports will be higher than with real sandbox data —  
that is expected and honest.

> If you have a Cuckoo / CAPEv2 / ANY.RUN sandbox report, see **Appendix A** for a  
> direct field mapping — skip this section entirely and use those values instead.

In [ ]:
# ── 2a. Load the trained scaler to get mean_ and scale_ ──────────────────────
# This gives us the exact distribution the behavioural agent was trained on.

sys.path.insert(0, str(Path('.').resolve()))
from Agentic_SABRE.agents.behavioural import BehavioralAgent

_beh_agent  = BehavioralAgent(T=1, device='cpu')
sc_mean     = _beh_agent.scaler.mean_    # shape (16,)
sc_scale    = _beh_agent.scaler.scale_   # shape (16,)

BEHAVIOR_COLUMNS = [
    "avg_read_throughput", "avg_write_throughput",
    "read_LBA_variance",   "write_LBA_variance",
    "write_entropy_mean",
    "mem_read_entropy_mean", "mem_write_entropy_mean", "mem_readwrite_entropy_mean",
    "mem_read_GPA_variance", "mem_write_GPA_variance",
    "mem_readwrite_GPA_variance", "mem_exec_GPA_variance",
    "mem_read_count", "mem_write_count", "mem_readwrite_count", "mem_exec_count",
]

print("Scaler statistics (what the model expects):")
diag = pd.DataFrame({'feature': BEHAVIOR_COLUMNS, 'mean': sc_mean, 'std': sc_scale})
diag['mean_exp'] = diag['mean'].apply(lambda x: f'{x:.3e}')
diag['std_exp']  = diag['std'].apply(lambda x: f'{x:.3e}')
print(diag[['feature','mean_exp','std_exp']].to_string(index=False))

In [ ]:
# ── 2b. Compute PE-derived signals (all normalised to 0–1 or small integers) ──

file_size       = len(raw_bytes)
sec_entropies   = [s['entropy']     for s in sections]
exec_entropies  = [s['entropy']     for s in sections if s['executable']]
write_entropies = [s['entropy']     for s in sections if s['writable']]
rw_entropies    = [s['entropy']     for s in sections if s['executable'] or s['writable']]
write_sections  = [s               for s in sections if s['writable']]
read_sections   = [s               for s in sections if not s['writable']]

va_all   = [float(s['virt_addr'])   for s in sections]
va_write = [float(s['virt_addr'])   for s in sections if s['writable']]
va_exec  = [float(s['virt_addr'])   for s in sections if s['executable']]
offsets  = [float(s['file_offset']) for s in sections]

mean_entropy  = float(np.mean(sec_entropies))   if sec_entropies  else 4.0
mean_w_ent    = float(np.mean(write_entropies)) if write_entropies else mean_entropy
mean_rw_ent   = float(np.mean(rw_entropies))    if rw_entropies   else mean_entropy
mean_ex_ent   = float(np.mean(exec_entropies))  if exec_entropies else mean_entropy

# API category counts
MEMORY_READ_APIS  = {'ReadFile','ReadFileEx','NtReadFile','ZwReadFile',
                     'ReadProcessMemory','VirtualQuery','VirtualQueryEx',
                     'GetModuleInformation','MapViewOfFile','MapViewOfFileEx'}
MEMORY_WRITE_APIS = {'WriteFile','WriteFileEx','NtWriteFile','ZwWriteFile',
                     'WriteProcessMemory','VirtualAlloc','VirtualAllocEx',
                     'HeapAlloc','LocalAlloc','GlobalAlloc'}
MEMORY_RW_APIS    = {'CreateFileMappingW','CreateFileMappingA','OpenFileMappingW',
                     'MapViewOfFile','CopyFileW','CopyFileA','MoveFileW','MoveFileA'}
MEMORY_EXEC_APIS  = {'CreateProcessW','CreateProcessA','ShellExecuteW','ShellExecuteA',
                     'WinExec','CreateRemoteThread','NtCreateThreadEx',
                     'VirtualProtect','VirtualProtectEx',
                     'LoadLibraryW','LoadLibraryA','LoadLibraryExW'}

n_read  = len(import_set & MEMORY_READ_APIS)
n_write = len(import_set & MEMORY_WRITE_APIS)
n_rw    = len(import_set & MEMORY_RW_APIS)
n_exec  = len(import_set & MEMORY_EXEC_APIS)

print(f"Section entropy — mean:{mean_entropy:.2f}  write:{mean_w_ent:.2f}  exec:{mean_ex_ent:.2f}")
print(f"API counts       — read:{n_read}  write:{n_write}  rw:{n_rw}  exec:{n_exec}")

In [ ]:
# ── 2c. Map each PE signal to a z-score ───────────────────────────────────────
#
# Rationale for each mapping:
#
# avg_read_throughput  (mean~40 GB/s): Large file / many readable sections
#   → more data to read → higher throughput. Normalised file size as proxy.
#   z ∈ [-1.5, +1.5]: small PE (100 KB) → z=-1.5, large PE (50 MB) → z=+1.5
#
# avg_write_throughput (mean~80 GB/s): Ransomware writes heavily.
#   High write entropy + many writable sections → higher z.
#
# read/write_LBA_variance (mean~8e16): High variance = random disk access
#   pattern (ransomware scans entire filesystem). More sections spread across
#   the file → higher offset variance → higher z.
#
# write_entropy_mean (mean~4090, std~30): The scaler mean ≈ 4096 suggesting
#   the raw feature is entropy × 512 (bits-per-kilobyte, not bits-per-byte).
#   Shannon entropy 5.0 → raw ≈ 2560; 7.5 → raw ≈ 3840; 8.0 → raw ≈ 4096.
#   Ransomware writes encrypted data at entropy ≈ 7.5–8.0 (near-random).
#   We map mean_write_entropy → raw directly using × 512.
#
# mem_*_entropy_mean (means are large negative ~-6e10): These are NOT 0-8 entropy.
#   They appear to be signed cumulative statistics from the kernel driver.
#   We treat the scaler mean as the baseline (z=0) and use section entropy
#   as a signed perturbation: high entropy → z positive.
#
# mem_*_GPA_variance (mean~5-8e16): Virtual address spread. More sections
#   at widely separated VAs → higher GPA variance in practice.
#   We normalise the VA variance from sections relative to scaler mean.
#
# mem_*_count (means ~10k, ~10k, ~7, ~62k): API import count as proxy.
#   We map n_apis / expected_saturation → z, where saturation means
#   "all common APIs of that type present".

def clamp(v, lo=-3.0, hi=3.0):
    return max(lo, min(hi, v))

# Reference file size for "average" sample: 2 MB
REF_SIZE = 2_000_000
size_z   = clamp((file_size / REF_SIZE - 1.0), -2, 2)

# Writable content fraction — higher means more write activity
write_frac = (sum(s['raw_size'] for s in write_sections) / max(file_size, 1))

# Entropy z-scores: 5.0 = neutral, each 1.5 bits = 1σ
# (Based on RanSMAP distribution: benign ~4-5 bits, ransomware ~7-8 bits)
ENTROPY_MID   = 5.0
ENTROPY_SIGMA = 1.5

ent_z_all  = clamp((mean_entropy  - ENTROPY_MID) / ENTROPY_SIGMA)
ent_z_writ = clamp((mean_w_ent    - ENTROPY_MID) / ENTROPY_SIGMA)
ent_z_rw   = clamp((mean_rw_ent   - ENTROPY_MID) / ENTROPY_SIGMA)
ent_z_exec = clamp((mean_ex_ent   - ENTROPY_MID) / ENTROPY_SIGMA)

# LBA / GPA variance z-scores from section offset/VA spread
# Normalise measured variance against the scaler's own std (scale_)**
offset_var = float(np.var(offsets)) if len(offsets) > 1 else 0.0
va_var_all = float(np.var(va_all))  if len(va_all) > 1  else 0.0
va_var_w   = float(np.var(va_write)) if len(va_write) > 1 else 0.0
va_var_ex  = float(np.var(va_exec))  if len(va_exec)  > 1 else 0.0

# scale these by the scaler's own std so the unit is comparable
lba_z  = clamp(offset_var / max(sc_scale[2], 1) - 1.0)
gpa_z  = clamp(va_var_all  / max(sc_scale[8], 1) - 1.0)
gpa_wz = clamp(va_var_w    / max(sc_scale[9], 1) - 1.0)
gpa_ez = clamp(va_var_ex   / max(sc_scale[11], 1) - 1.0)

# Count z-scores: saturation at 5 APIs of each type → z ≈ +1
COUNT_SAT = 5.0
rd_z  = clamp(n_read  / COUNT_SAT - 1.0)
wr_z  = clamp(n_write / COUNT_SAT - 1.0)
rw_z  = clamp(n_rw    / COUNT_SAT - 1.0) * 0.5  # smaller range (mean≈7)
ex_z  = clamp(n_exec  / COUNT_SAT - 1.0)

z_scores = [
    size_z,          # avg_read_throughput
    ent_z_writ,      # avg_write_throughput  (entropy-heavy writes = high tp)
    lba_z,           # read_LBA_variance
    lba_z,           # write_LBA_variance  (symmetric proxy)
    None,            # write_entropy_mean  → direct conversion below
    ent_z_all,       # mem_read_entropy_mean
    ent_z_writ,      # mem_write_entropy_mean
    ent_z_rw,        # mem_readwrite_entropy_mean
    gpa_z,           # mem_read_GPA_variance
    gpa_wz,          # mem_write_GPA_variance
    gpa_z,           # mem_readwrite_GPA_variance
    gpa_ez,          # mem_exec_GPA_variance
    rd_z,            # mem_read_count
    wr_z,            # mem_write_count
    rw_z,            # mem_readwrite_count
    ex_z,            # mem_exec_count
]

print("Z-scores derived from PE:")
for col, z in zip(BEHAVIOR_COLUMNS, z_scores):
    print(f"  {col:<36}: {f'{z:.3f}' if z is not None else 'direct'}")

In [ ]:
# ── 2d. Reconstruct raw values in the scaler's space ─────────────────────────
#
# raw = mean + z * scale
#
# Special case — write_entropy_mean:
# The scaler mean ≈ 4090 and std ≈ 30.  The feature appears to be stored as
# entropy × 512 (i.e. bits per kilobyte instead of bits per byte).
# We convert the PE section entropy directly:
#   raw = mean_write_entropy × 512
# This is more faithful than a z-score guess for this one feature.

beh_raw = np.empty(16, dtype=np.float64)

for i, (col, z) in enumerate(zip(BEHAVIOR_COLUMNS, z_scores)):
    if col == 'write_entropy_mean':
        beh_raw[i] = mean_w_ent * 512.0          # direct unit conversion
    else:
        beh_raw[i] = sc_mean[i] + z * sc_scale[i]

beh_scaled = beh_raw.astype(np.float32)

# ── Diagnostic: show normalised (z-score) form so you can see where values land
z_check = (beh_scaled - sc_mean) / sc_scale

diag_df = pd.DataFrame({
    'feature':   BEHAVIOR_COLUMNS,
    'raw_value': [f'{v:.3e}' for v in beh_scaled],
    'z_score':   [f'{z:.3f}' for z in z_check],
    'scaler_mean': [f'{m:.3e}' for m in sc_mean],
})
print(diag_df.to_string(index=False))
print()
print("Z-scores near 0 are 'average sample'. Positive = more suspicious, negative = more benign.")
print("Values beyond ±2 are extrapolation — treat agent output as uncertain.")

In [ ]:
# ── 2e. Final behavioural vector ──────────────────────────────────────────────

x_beh = {"features": beh_scaled}

print(f"x_beh shape : {x_beh['features'].shape}  dtype={x_beh['features'].dtype}")
print(f"Value range : [{beh_scaled.min():.3e}, {beh_scaled.max():.3e}]")

---
## 3 · Knowledge Feature Extraction (21-D)

MITRE ATT&CK tactic counts from import-to-tactic mapping,  
known-tool detection from PE string signatures, and platform flags from PE headers.

In [ ]:
# ── 3a. MITRE ATT&CK tactic → API mapping ────────────────────────────────────

TACTIC_API_MAP = {
    'Tactic_Initial_Access':        {'URLDownloadToFileW','URLDownloadToFileA','InternetOpenW',
                                     'InternetOpenA','HttpOpenRequestW','WinHttpOpen',
                                     'FtpOpenFileW','ShellExecuteW','ShellExecuteA'},
    'Tactic_Execution':             {'CreateProcessW','CreateProcessA','WinExec','ShellExecuteW',
                                     'CreateRemoteThread','NtCreateThreadEx','RtlCreateUserThread',
                                     'CreateFiber','ConvertThreadToFiber'},
    'Tactic_Persistence':           {'RegSetValueExW','RegSetValueExA','RegCreateKeyExW',
                                     'RegCreateKeyExA','CreateServiceW','CreateServiceA',
                                     'StartServiceW','SHGetSpecialFolderPathW',
                                     'SHGetFolderPathW','WritePrivateProfileStringW'},
    'Tactic_Privilege_Escalation':  {'AdjustTokenPrivileges','LookupPrivilegeValueW',
                                     'OpenProcessToken','DuplicateTokenEx',
                                     'ImpersonateLoggedOnUser','SetTokenInformation',
                                     'NtSetSystemInformation'},
    'Tactic_Credential_Access':     {'CryptAcquireContextW','CryptHashData','CryptDeriveKey',
                                     'SamOpenDatabase','SamQueryInformationUser',
                                     'LsaOpenPolicy','LsaRetrievePrivateData',
                                     'CredReadW','CredEnumerateW'},
    'Tactic_Lateral_Movement':      {'WNetOpenEnumW','WNetEnumResourceW','WNetAddConnection2W',
                                     'NetShareEnum','NetServerEnum',
                                     'RpcBindingFromStringBindingW','CoCreateInstance'},
    'Tactic_Discovery':             {'FindFirstFileW','FindFirstFileA','FindNextFileW',
                                     'FindNextFileA','GetDriveTypeW','GetLogicalDrives',
                                     'GetVolumeInformationW','EnumProcesses',
                                     'CreateToolhelp32Snapshot','Process32FirstW',
                                     'RegOpenKeyExW','RegEnumKeyExW','RegEnumValueW'},
    'Tactic_Command_and_Control':   {'WSAStartup','connect','send','recv',
                                     'HttpSendRequestW','HttpSendRequestA',
                                     'WinHttpSendRequest','WinHttpReceiveResponse',
                                     'InternetConnectW','InternetReadFile'},
    'Tactic_Exfiltration':          {'FtpPutFileW','FtpPutFileA','InternetWriteFile',
                                     'WinHttpSendRequest','CopyFileW','CopyFileA','MoveFileW'},
    'Tactic_Impact':                {'CryptEncrypt','BCryptEncrypt','CryptGenKey',
                                     'BCryptGenerateSymmetricKey','DeleteFileW','DeleteFileA',
                                     'DeviceIoControl','SetEndOfFile','SetFilePointerEx',
                                     'CreateVssBackupComponents'},
}

tactic_counts = {t: len(import_set & apis) for t, apis in TACTIC_API_MAP.items()}
tactic_total  = sum(tactic_counts.values())

for t, c in tactic_counts.items():
    print(f"  {t:<40}: {c}")

In [ ]:
# ── 3b. Known-tool string signature matching ──────────────────────────────────

TOOL_SIGNATURES = {
    'Tool_Cobalt_Strike': ['beacon','cobaltstrike','cobalt strike','sleep_mask',
                           'pipe_name','c2_server','post_ex',
                           r'MSSE-\d+-server',r'msagent_\w+'],
    'Tool_Mimikatz':      ['mimikatz','sekurlsa','kerberos::','wdigest','lsadump',
                           'logonpasswords','privilege::debug'],
    'Tool_PsExec':        ['psexec','paexec','PSEXESVC','psexecsvc'],
    'Tool_Rclone':        ['rclone','rclone.exe','--config rclone','rclone copy','mega:'],
    'Tool_MegaNZ':        ['mega.nz','mega.co.nz','megacmd','MEGAclient'],
    'Tool_AnyDesk':       ['anydesk','AnyDesk.exe','ad_remotesession'],
    'Tool_TeamViewer':    ['teamviewer','TeamViewer.exe','tvnserver','TV_W32_Service'],
    'Tool_BloodHound':    ['bloodhound','sharphound','SharpHound','invoke-bloodhound'],
}

strings_blob = ' '.join(s.lower() for s in all_strings)

tool_counts = {}
for tool, patterns in TOOL_SIGNATURES.items():
    tool_counts[tool] = sum(len(re.findall(p.lower(), strings_blob)) for p in patterns)

tool_total = sum(tool_counts.values())

for t, c in tool_counts.items():
    print(f"  {t:<30}: {c}")

In [ ]:
# ── 3c. Platform detection + final 21-D vector ────────────────────────────────

has_windows = pe.FILE_HEADER.Machine in (0x8664, 0x014c)
has_linux   = any(s in strings_blob for s in ['linux','/proc/','/sys/','bash','/bin/sh'])
has_esxi    = any(s in strings_blob for s in ['esxi','vmkernel','.vmdk','esxcfg-'])

# Tool_Total_Mentions / Tactic_Total_Mentions define the training label and are not
# model inputs (Agentic_SABRE/knowledge_features.py).
KNOWLEDGE_COLUMNS = [
    "Tactic_Initial_Access","Tactic_Execution","Tactic_Persistence",
    "Tactic_Privilege_Escalation","Tactic_Credential_Access",
    "Tactic_Lateral_Movement","Tactic_Discovery",
    "Tactic_Command_and_Control","Tactic_Exfiltration","Tactic_Impact",
    "Tool_Cobalt_Strike","Tool_Mimikatz","Tool_PsExec",
    "Tool_Rclone","Tool_MegaNZ","Tool_AnyDesk",
    "Tool_TeamViewer","Tool_BloodHound",
    "Platform_Windows","Platform_Linux","Platform_ESXi",
]

know_values = np.array([
    *[float(tactic_counts[f'Tactic_{t}']) for t in [
        'Initial_Access','Execution','Persistence','Privilege_Escalation',
        'Credential_Access','Lateral_Movement','Discovery',
        'Command_and_Control','Exfiltration','Impact']],
    *[float(tool_counts[f'Tool_{t}']) for t in [
        'Cobalt_Strike','Mimikatz','PsExec','Rclone',
        'MegaNZ','AnyDesk','TeamViewer','BloodHound']],
    float(has_windows), float(has_linux), float(has_esxi),
], dtype=np.float32)

x_know = {"features": know_values}

print(pd.Series(know_values, index=KNOWLEDGE_COLUMNS).to_string())

---
## 4 · Summary and orchestrator hand-off

In [ ]:
# ── 4a. Summary ───────────────────────────────────────────────────────────────
print("=" * 60)
print(f"File           : {EXE_PATH.name}")
print(f"MD5            : {md5}")
print(f"SHA-256        : {sha256}")
print("-" * 60)
print(f"x_sem  shape   : {x_sem['embedding'].shape}")
print(f"x_beh  shape   : {x_beh['features'].shape}")
print(f"x_know shape   : {x_know['features'].shape}")
print("-" * 60)
sus_count = len([api for api in SUSPICIOUS_APIS if api in import_set])
print(f"Suspicious APIs found : {sus_count} / {len(SUSPICIOUS_APIS)}")
print(f"Tactic signals        : {tactic_total}")
print(f"Tool markers          : {tool_total}")
print(f"Max section entropy   : {max(sec_entropies):.3f}  (>7.2 = likely packed/encrypted)")
print("=" * 60)

In [ ]:
# ── 4b. Run through the live orchestrator ─────────────────────────────────────

from Agentic_SABRE.orchestrator_3agent import Orchestrator

orch   = Orchestrator(decision_artifact="checkpoints/decision_fuser_3agent.json")
result = orch.handle(x_sem, x_beh, x_know)

print(f"Decision        : {result['decision']}")
print(f"Fused score     : {result['score_fused']:.4f}")
print(f"Max uncertainty : {result['uncert_max']:.4f}")
print()
for agent, data in result.get('by_agent', {}).items():
    print(f"  {agent:<12}: p={data.get('prob',0):.4f}  σ={data.get('uncert',0):.4f}")

In [ ]:
# ── 4c. Save to CSV ───────────────────────────────────────────────────────────

OUT_DIR = Path("extracted_features")
OUT_DIR.mkdir(exist_ok=True)
stem = EXE_PATH.stem

pd.DataFrame([sem_embedding], columns=[f"emb_{i}" for i in range(384)]
    ).to_csv(OUT_DIR / f"{stem}_semantic.csv",     index=False)
pd.DataFrame([beh_scaled],   columns=BEHAVIOR_COLUMNS
    ).to_csv(OUT_DIR / f"{stem}_behavioural.csv",  index=False)
pd.DataFrame([know_values],  columns=KNOWLEDGE_COLUMNS
    ).to_csv(OUT_DIR / f"{stem}_knowledge.csv",    index=False)

print(f"Saved to {OUT_DIR}/")

---
## Appendix A — Replacing behavioural proxies with real sandbox values

If you have a Cuckoo / CAPEv2 / ANY.RUN report, extract the values below and  
pass them directly to `x_beh` — skip Section 2 entirely.

| SABRE feature | Cuckoo / CAPE JSON path |
|---|---|
| `avg_read_throughput` | sum of `ReadFile.buffer_size` calls / execution duration (s) |
| `avg_write_throughput` | sum of `WriteFile.buffer_size` calls / execution duration (s) |
| `read_LBA_variance` | `np.var([call.offset for call in ReadFile calls])` |
| `write_LBA_variance` | `np.var([call.offset for call in WriteFile calls])` |
| `write_entropy_mean` | mean Shannon entropy of each `WriteFile` buffer × 512 |
| `mem_*_entropy_mean` | mean entropy of memory regions per access type (driver log) |
| `mem_*_GPA_variance` | `np.var([region.base_address for each access type])` |
| `mem_*_count` | total call count per memory access type |

---
## Appendix B — Why `mem_*_entropy_mean` values are negative

The scaler means for these features are large negative numbers (~-6×10^10).  
This is because the RanSMAP driver records **raw signed 64-bit counter values**  
from the hypervisor, not normalised 0–8 entropy. The sign and magnitude depend  
on the internal counter offset. The StandardScaler was fitted on this raw scale,  
so by anchoring to the mean (z=0) and perturbing by ±1σ based on PE entropy,  
we stay within the distribution the model has seen.